In [13]:
# 2025.12.01 Ensemble Set NO.5 돌리기

In [14]:
import sys

project_root = 'c:/big20/git/big20-ML-project2-team3/CreditCardFraud'

# sys.path에 추가 (모듈 import용)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [15]:
import os

import time
import pandas as pd
import numpy  as np
import matplotlib.pyplot as plt
import seaborn as sns


import warnings
warnings.filterwarnings('ignore')

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.metrics         import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics         import classification_report
from sklearn.metrics         import roc_auc_score
from sklearn.metrics         import precision_recall_curve, classification_report
from sklearn.datasets        import make_classification
from sklearn.preprocessing   import RobustScaler

# Model import
from sklearn.tree           import DecisionTreeClassifier
from sklearn.ensemble       import StackingClassifier
from sklearn.ensemble       import RandomForestClassifier
from sklearn.ensemble       import GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model   import LogisticRegression
from sklearn.linear_model   import LinearRegression
from sklearn.linear_model   import SGDClassifier
from sklearn.svm            import SVC
from sklearn.svm            import LinearSVC
from xgboost                import XGBClassifier
from xgboost                import plot_importance
from lightgbm               import LGBMClassifier
from catboost               import CatBoostClassifier
from sklearn.ensemble       import HistGradientBoostingClassifier
from sklearn.calibration    import CalibratedClassifierCV # stacking시 LSVC 에러용

# hyperopt 용
from hyperopt               import hp

# 사용자 Functions import
import HyperParams          as HP 
import utils.data_sampling  as ds 

import importlib

from utils import user_utils
importlib.reload(user_utils)

from utils import user_utils    as uu
from utils import preprocessing as pp
from utils import data_sampling as ds
from utils import model_utils   as mu
from utils import modeling      as mo

from utils.hyperopt_search import hyperopt_search, train_and_evaluate

In [16]:
# -------------------------------------------------------
# 🔷 모델 생성 함수 (HyperOpt 파라미터 자동 적용)
# -------------------------------------------------------

def get_models():
    models = {
        "cat": CatBoostClassifier(**HP.cb_best_params),
        "dt": DecisionTreeClassifier(**HP.dt_basic_params),
        "gb": GradientBoostingClassifier(**HP.gb_best_params),
        "hgb": HistGradientBoostingClassifier(**HP.hgb_best_params),
        "lgbm": LGBMClassifier(**HP.lgbm_best_param2),
        "lr": LogisticRegression(**HP.lr_best_params),
        "lir" : LinearRegression(**HP.lir_best_params),
        "lsvc": LinearSVC(**HP.lsvc_best_params),
        "mlp": MLPClassifier(**HP.mlp_basic_params),
        "rf": RandomForestClassifier(**HP.rf_best_params),
        "sgd": SGDClassifier(**HP.sgd_best_params),
        "svm_rbf": SVC(**HP.svc_rbf_best_params),
        "xgb": XGBClassifier(**HP.xgb_best_params, eval_metric='logloss'),
    }

    return models
# eof ----------------------------------------------------------------

In [17]:
# 결과받을 딕셔너리
results = {}
team_rs = 23 # 우리팀 random_state

In [18]:
#1. 데이터 로딩
raw_df = pp.ccf_load_data()

데이터 로드 성공: (284807, 31)


In [19]:
# 2. 데이터 전처리
# 2.1 Time 컬럼 삭제 , 데이터,타겟 분리
X_features, y_target = pp.split_features_target(raw_df, cols= 'Time')
X_features.shape, y_target.shape

((284807, 29), (284807,))

In [20]:
# 2.2 이상치를 경계값으로 치환
cap_X_feature = pp.cap_outliers(X_features)

In [21]:
# 2.3 학습/테스트 데이터 분리
X_train, X_test, y_train, y_test = pp.data_split(cap_X_feature, y_target)

In [22]:
# 학습/검증 데이터 분리
# X_tr, X_val, y_tr, y_val = pp.data_split(X_train, y_train, size=0.4)

In [23]:
# 2.4 Over Sampling 하는 경우
X_over, y_over = ds.oversampling_smote(X_train, y_train)


✅ SMOTE 오버샘플링 완료
   원본 샘플 수: 227845 (Class 0: 227451, Class 1: 394)
   샘플링 후: 454902 (Class 0: 227451, Class 1: 227451)


In [24]:
# Over Sampling한 경우 학습/검증 데이터 분리
# X_tr_over, X_val_over, y_tr_over, y_val_over = pp.data_split(X_over, y_over, size=0.4)

In [25]:
# 사용자 Functions import - 에러나서 다시 실행
# import importlib
# from utils import hyperopt_search
# importlib.reload(hyperopt_search)

# from utils.hyperopt_search import hyperopt_search, train_and_evaluate

In [26]:
# BestOpt 찾고 나서 스케일적용 버전 만들어서 모델링하기 
# 2.5 StandardScaler 적용
X_train_sscaled, X_test_sscaled, scaler = pp.scale_data(X_train, X_test)


In [27]:
# 2.6 robustScaler 적용

rscaler = RobustScaler()
X_train_rscaled = rscaler.fit_transform(X_train)   # 학습 데이터로 fit + transform
X_test_rscaled = rscaler.transform(X_test)         # 테스트 데이터는 transform만


### ChatGpt 추천 조합

| 조합 번호 | 구성                         | 특징              |
| ----- | -------------------------- | --------------- |
| **1** | CatBoost + XGB + LGBM + LR | 성능 최상위 전천후      |
| **2** | LGBM + RF + MLP            | 구조적 다양성 최고      |
| **3** | CatBoost + GB + SVM(RBF)   | Recall 최적화      |
| **4** | XGB + LR + SVM(linear)     | 단순·안정·일관된 결정 경계 |
| **5** | RF + SGD + MLP + LR        | 전통 ML 메타 스택     |

In [28]:
def create_lr(best_params):
    """
    LogisticRegression 모델을 HyperOpt/Optuna로 찾은 best_params 기반으로 생성하는 함수.

    이 함수는 LogisticRegression의 solver와 penalty 조합이 유효한지 검사하고,
    유효하지 않은 조합이 들어올 경우 자동으로 수정하여 안전하게 모델을 생성한다.

    Parameters
    ----------
    best_params : dict
        HyperOpt 또는 Optuna로 최적화한 LogisticRegression의 최적 파라미터 딕셔너리.
        예: {"C": 0.1, "solver": "liblinear", "penalty": "l1"}

    Returns
    -------
    LogisticRegression
        최종적으로 검증된 파라미터로 생성된 LogisticRegression 모델.

    Notes
    -----
    - solver에 따라 사용할 수 있는 penalty 종류가 다르기 때문에,
      최적화 결과가 잘못된 조합을 반환할 가능성이 있음.
    - 안전성을 위해 직접 검증한 후 잘못된 penalty는 'l2'로 자동 변경한다.
    - 변경이 발생하면 경고 메시지를 출력한다.
    """

    # 최적화된 solver, penalty 값을 가져오고 기본값 설정
    solver = best_params.get('solver', 'liblinear')
    penalty = best_params.get('penalty', 'l2')

    # solver별로 허용되는 penalty 목록 정의
    valid_penalties = {
        'liblinear': ['l1', 'l2'],
        'lbfgs': ['l2', 'none'],
        'saga': ['l1', 'l2', 'elasticnet', 'none'],
        'newton-cg': ['l2', 'none'],
    }

    # penalty가 solver에 맞지 않으면 자동 수정
    if penalty not in valid_penalties.get(solver, []):
        print(f"[WARN] penalty '{penalty}' is incompatible with solver '{solver}'. Using 'l2'")
        best_params['penalty'] = 'l2'

    # 유효한 파라미터로 LogisticRegression 모델 생성
    return LogisticRegression(**best_params)
# eof -----------------------------------------------------------


In [29]:
# ======================================
# 🔷 Stacking #1: CatBoost + XGB + LGBM
# ======================================

models = get_models()

estimators_1 = [
    ('cat', models['cat']),
    ('xgb', models['xgb']),
    ('lgbm', models['lgbm']),
]


stack_1 = StackingClassifier(
    estimators=estimators_1,
    final_estimator=create_lr(HP.lr_best_params),
    stack_method='predict_proba',
    cv=5,
    n_jobs=-1
)
option_name = 'stack1(cat+xgb+lgbm)_ho_best'
results1 = uu.get_model_train_eval(stack_1, f'{option_name}', X_train, X_test, y_train, y_test)



🚀 모델 학습 시작: stack1(cat+xgb+lgbm)_ho_best


[LightGBM] [Warning] min_data_in_leaf is set=25, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=25
[LightGBM] [Warning] feature_fraction is set=0.801203, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.801203
[LightGBM] [Warning] min_gain_to_split is set=0.002445, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=0.002445
[LightGBM] [Warning] lambda_l1 is set=0.512937, reg_alpha=0.548894 will be ignored. Current value: lambda_l1=0.512937
[LightGBM] [Warning] lambda_l2 is set=1.092e-07, reg_lambda=2.435e-08 will be ignored. Current value: lambda_l2=1.092e-07
[LightGBM] [Warning] bagging_fraction is set=0.825535, subsample=1.0 will be ignored. Current value: bagging_fraction=0.825535
[LightGBM] [Warning] bagging_freq is set=4, subsample_freq=0 will be ignored. Current value: bagging_freq=4
[LightGBM] [Warning] min_data_in_leaf is set=25, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=25
[LightGBM] [W

folder = c:\big20\git\big20-ML-project2-team3\CreditCardFraud\results
{'result_dict': {'AUC': 0.9726, '정확도': 0.9876, '정밀도': 0.1105, '재현율': 0.8776, 'F1': 0.1963, 'F2': 0.3675}, '오차행렬': [[56172, 692], [12, 86]], '실행 시간': 69.5844}

📊 Base Estimators 평가 중 (3개)...


[LightGBM] [Warning] min_data_in_leaf is set=25, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=25
[LightGBM] [Warning] feature_fraction is set=0.801203, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.801203
[LightGBM] [Warning] min_gain_to_split is set=0.002445, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=0.002445
[LightGBM] [Warning] lambda_l1 is set=0.512937, reg_alpha=0.548894 will be ignored. Current value: lambda_l1=0.512937
[LightGBM] [Warning] lambda_l2 is set=1.092e-07, reg_lambda=2.435e-08 will be ignored. Current value: lambda_l2=1.092e-07
[LightGBM] [Warning] bagging_fraction is set=0.825535, subsample=1.0 will be ignored. Current value: bagging_fraction=0.825535
[LightGBM] [Warning] bagging_freq is set=4, subsample_freq=0 will be ignored. Current value: bagging_freq=4
[LightGBM] [Warning] min_data_in_leaf is set=25, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=25
[LightGBM] [W

stack1(cat+xgb+lgbm)_ho_best - 저장 중: 100%|█████████████████████████| 4/4 [01:10<00:00, 17.61s/it]

✓ 모델 저장 완료: ../models\stack1(cat+xgb+lgbm)_ho_best.pkl
  파일 크기: 1.03 MB

✅ 완료: stack1(cat+xgb+lgbm)_ho_best (실행시간: 69.58초)



In [30]:
# ======================================
# 🔷 Stacking #2: LGBM + RF + MLP
# ======================================

models = get_models()

estimators_2 = [
    ('lgbm', models['lgbm']),
    ('rf', models['rf']),
    ('mlp', models['mlp']),
]

stack_2 = StackingClassifier(
    estimators=estimators_2,
    final_estimator=create_lr(HP.lr_best_params),
    stack_method='predict_proba',
    cv=5,
    n_jobs=-1
)
#X_over, y_over
option_name = 'stack2(lgbm+rf+mlp)_ho_best_smote'
results2 = uu.get_model_train_eval(stack_2, f'{option_name}', X_over, X_test, y_over, y_test)



🚀 모델 학습 시작: stack2(lgbm+rf+mlp)_ho_best_smote


[LightGBM] [Warning] min_data_in_leaf is set=25, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=25
[LightGBM] [Warning] feature_fraction is set=0.801203, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.801203
[LightGBM] [Warning] min_gain_to_split is set=0.002445, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=0.002445
[LightGBM] [Warning] lambda_l1 is set=0.512937, reg_alpha=0.548894 will be ignored. Current value: lambda_l1=0.512937
[LightGBM] [Warning] lambda_l2 is set=1.092e-07, reg_lambda=2.435e-08 will be ignored. Current value: lambda_l2=1.092e-07
[LightGBM] [Warning] bagging_fraction is set=0.825535, subsample=1.0 will be ignored. Current value: bagging_fraction=0.825535
[LightGBM] [Warning] bagging_freq is set=4, subsample_freq=0 will be ignored. Current value: bagging_freq=4
[LightGBM] [Warning] min_data_in_leaf is set=25, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=25
[LightGBM] [W

folder = c:\big20\git\big20-ML-project2-team3\CreditCardFraud\results
{'result_dict': {'AUC': 0.9654, '정확도': 0.9994, '정밀도': 0.7961, '재현율': 0.8367, 'F1': 0.8159, 'F2': 0.8283}, '오차행렬': [[56843, 21], [16, 82]], '실행 시간': 2835.4283}

📊 Base Estimators 평가 중 (3개)...


[LightGBM] [Warning] min_data_in_leaf is set=25, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=25
[LightGBM] [Warning] feature_fraction is set=0.801203, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.801203
[LightGBM] [Warning] min_gain_to_split is set=0.002445, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=0.002445
[LightGBM] [Warning] lambda_l1 is set=0.512937, reg_alpha=0.548894 will be ignored. Current value: lambda_l1=0.512937
[LightGBM] [Warning] lambda_l2 is set=1.092e-07, reg_lambda=2.435e-08 will be ignored. Current value: lambda_l2=1.092e-07
[LightGBM] [Warning] bagging_fraction is set=0.825535, subsample=1.0 will be ignored. Current value: bagging_fraction=0.825535
[LightGBM] [Warning] bagging_freq is set=4, subsample_freq=0 will be ignored. Current value: bagging_freq=4
[LightGBM] [Warning] min_data_in_leaf is set=25, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=25
[LightGBM] [W




stack2(lgbm+rf+mlp)_ho_best_smote - 저장 중: 100%|███████████████████| 4/4 [47:16<00:00, 709.19s/it]

✓ 모델 저장 완료: ../models\stack2(lgbm+rf+mlp)_ho_best_smote.pkl
  파일 크기: 9.17 MB

✅ 완료: stack2(lgbm+rf+mlp)_ho_best_smote (실행시간: 2835.43초)



In [31]:
# ======================================
# 🔷 Stacking #3: CatBoost + GB + SVM(RBF)
# ======================================

models = get_models()

estimators_3 = [
    ('cat', models['cat']),
    ('gb', models['gb']),
    ('svm_rbf', models['svm_rbf']),
]

stack_3 = StackingClassifier(
    estimators=estimators_3,
    final_estimator=create_lr(HP.lr_best_params),
    stack_method='predict_proba',
    cv=5,
    n_jobs=-1
)

option_name = 'cat+gb+svm_rbf_ho_best'

results[option_name] = uu.get_model_train_eval(stack_3, f'{option_name}', X_train_sscaled, X_test_sscaled, y_train, y_test)


🚀 모델 학습 시작: cat+gb+svm_rbf_ho_best


folder = c:\big20\git\big20-ML-project2-team3\CreditCardFraud\results
{'result_dict': {'AUC': 0.9704, '정확도': 0.9956, '정밀도': 0.2633, '재현율': 0.8571, 'F1': 0.4029, 'F2': 0.5907}, '오차행렬': [[56629, 235], [14, 84]], '실행 시간': 10587.879}

📊 Base Estimators 평가 중 (3개)...






cat+gb+svm_rbf_ho_best - 저장 중: 100%|███████████████████████████| 4/4 [3:16:44<00:00, 2951.10s/it]

✓ 모델 저장 완료: ../models\cat+gb+svm_rbf_ho_best.pkl
  파일 크기: 5.98 MB

✅ 완료: cat+gb+svm_rbf_ho_best (실행시간: 10587.88초)



In [32]:
def create_calibrated_lsvc(model):
    """
    predict_proba를 지원하는 LinearSVC 생성
    
    CalibratedClassifierCV는 decision_function을 확률로 변환해줍니다.
    """
    base_svc = model
    calibrated_svc = CalibratedClassifierCV(base_svc, cv=3)
    return calibrated_svc

In [33]:
# ======================================
# 🔷 Stacking #4: XGB + LR + SVM(linear)
# ======================================
# LR은 메타모델로 사용하는 게 더 좋아서 base에는 넣지 않음
models = get_models()
est_lr = LogisticRegression(
    penalty='l2',
    C=1.0,
    solver='liblinear',
    class_weight='balanced'
)

estimators_4 = [
    ('xgb', models['xgb']),
    ('lsvc', create_calibrated_lsvc(models['lsvc'])),
]

stack_4 = StackingClassifier(
    estimators=estimators_4,
    final_estimator=est_lr,
    stack_method='predict_proba',
    cv=5,
    n_jobs=-1
)
option_name = 'stack4(xgb+lsvc+lr)_ho_best'
results4 = uu.get_model_train_eval(stack_4, f'{option_name}', X_train, X_test, y_train, y_test)




🚀 모델 학습 시작: stack4(xgb+lsvc+lr)_ho_best


folder = c:\big20\git\big20-ML-project2-team3\CreditCardFraud\results
{'result_dict': {'AUC': 0.9734, '정확도': 0.988, '정밀도': 0.1138, '재현율': 0.8776, 'F1': 0.2014, 'F2': 0.3746}, '오차행렬': [[56194, 670], [12, 86]], '실행 시간': 18.6208}

📊 Base Estimators 평가 중 (2개)...




stack4(xgb+lsvc+lr)_ho_best - 저장 중: 100%|██████████████████████████| 4/4 [00:18<00:00,  4.73s/it]

✓ 모델 저장 완료: ../models\stack4(xgb+lsvc+lr)_ho_best.pkl
  파일 크기: 0.17 MB

✅ 완료: stack4(xgb+lsvc+lr)_ho_best (실행시간: 18.62초)



In [34]:
# ======================================
# 🔷 Stacking #5: RF + SGD + MLP + LR
# ======================================
models = get_models()
estimators_5 = [
    ('rf', models['rf']),
    ('sgd', models['sgd']),
    ('mlp', models['mlp']),
]

stack_5 = StackingClassifier(
    estimators=estimators_5,
    final_estimator=create_lr(HP.lr_best_params),
    stack_method='predict_proba',
    cv=5,
    n_jobs=-1
)
option_name = 'stack5(rf+sgd+mlp+lr)_ho_best'
results = uu.get_model_train_eval(stack_5, f'{option_name}', X_train, X_test, y_train, y_test)


🚀 모델 학습 시작: stack5(rf+sgd+mlp+lr)_ho_best


folder = c:\big20\git\big20-ML-project2-team3\CreditCardFraud\results
{'result_dict': {'AUC': 0.97, '정확도': 0.9905, '정밀도': 0.1398, '재현율': 0.8776, 'F1': 0.2412, 'F2': 0.427}, '오차행렬': [[56335, 529], [12, 86]], '실행 시간': 944.042}

📊 Base Estimators 평가 중 (3개)...





stack5(rf+sgd+mlp+lr)_ho_best - 저장 중: 100%|███████████████████████| 4/4 [15:44<00:00, 236.16s/it]

✓ 모델 저장 완료: ../models\stack5(rf+sgd+mlp+lr)_ho_best.pkl
  파일 크기: 2.69 MB

✅ 완료: stack5(rf+sgd+mlp+lr)_ho_best (실행시간: 944.04초)



In [35]:
# # ===========================================
# # 🔷 5개 Stack 모델을 Soft Voting으로 결합
# # ===========================================

# from sklearn.ensemble import VotingClassifier

# # VotingClassifier는 원래 predict_proba를 지원하는 모델만 가능
# # 우리 스택 모델들은 모두 predict_proba 지원하므로 문제 없음
# weights_desc = '''
# | 스택                         | 추천 이유     | weight |
# | -------------------------- | --------- | ------ |
# | Stack #1 (Boosting+LR)     | 대부분 최고 성능 | **3**  |
# | Stack #3 (CatBoost+GB+SVM) | Recall 강함 | **2**  |
# | Stack #2 (LGBM+RF+MLP)     | 안정적       | **2**  |
# | Stack #4 (XGB+LR+SVM)      | 선형 경계 보정  | **1**  |
# | Stack #5 (RF+SGD+MLP)      | 편향 다양성 확보 | **1**  |
# '''

# voting_ensemble = VotingClassifier(
#     estimators=[
#         ('stack1', stack_1),
#         ('stack2', stack_2),
#         ('stack3', stack_3),
#         ('stack4', stack_4),
#         ('stack5', stack_5),
#     ],
#     voting='soft',          # 🔥 중요: 확률 기반 soft voting
#     weights=[3, 2, 2, 1, 1], # 가중치 
#     n_jobs=-1
# )

In [39]:
import numpy as np

# ==============================
# 1) 모델 이름 목록 (이미 저장된 모델)
# ==============================
stack_model_names = [
    'stack1(cat+xgb+lgbm)_ho_best',
    'stack2(lgbm+rf+mlp)_ho_best_smote',
    'cat+gb+svm_rbf_ho_best',
    'stack4(xgb+lsvc+lr)_ho_best',
    'stack5(rf+sgd+mlp+lr)_ho_best'
]

weights = [3, 3, 2, 2, 1]   # 네 설계 기준 가중치


# ==============================
# 2) 모델 로딩 + 예측 확률 생성
# ==============================
preds = []
print("📥 저장된 모델 기반 예측 생성 중...\n")

for i, name in enumerate(stack_model_names):
    model = mu.load_model(name)
    proba = model.predict_proba(X_test)[:, 1]   # 확률값만 가져옴
    preds.append(proba)
    print(f"✔ {name} → 예측 완료 (shape={proba.shape})")


# ==============================
# 3) Soft Voting (가중 평균)
# ==============================
preds = np.array(preds)
final_pred_proba = np.average(preds, axis=0, weights=weights)

# threshold 적용 (기본 0.5, 후에 조정 가능)
final_pred = (final_pred_proba > 0.5).astype(int)


# ==============================
# 4) 평가 출력
# ==============================
from sklearn.metrics import (
    roc_auc_score, confusion_matrix,
    precision_score, recall_score,
    f1_score, classification_report
)

print("\n📊 Final Soft Voting Performance")
print("----------------------------------")
print("AUC:", round(roc_auc_score(y_test, final_pred_proba), 4))
print("Precision:", round(precision_score(y_test, final_pred), 4))
print("Recall:", round(recall_score(y_test, final_pred), 4))
print("F1:", round(f1_score(y_test, final_pred), 4))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, final_pred))


📥 저장된 모델 기반 예측 생성 중...

✓ 모델 로드 완료: ../models\stack1(cat+xgb+lgbm)_ho_best.pkl
  모델 타입: StackingClassifier
[LightGBM] [Warning] min_data_in_leaf is set=25, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=25
[LightGBM] [Warning] feature_fraction is set=0.801203, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.801203
[LightGBM] [Warning] min_gain_to_split is set=0.002445, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=0.002445
[LightGBM] [Warning] lambda_l1 is set=0.512937, reg_alpha=0.548894 will be ignored. Current value: lambda_l1=0.512937
[LightGBM] [Warning] lambda_l2 is set=1.092e-07, reg_lambda=2.435e-08 will be ignored. Current value: lambda_l2=1.092e-07
[LightGBM] [Warning] bagging_fraction is set=0.825535, subsample=1.0 will be ignored. Current value: bagging_fraction=0.825535
[LightGBM] [Warning] bagging_freq is set=4, subsample_freq=0 will be ignored. Current value: bagging_freq=4
✔ stack1(cat+xgb+lgbm)_ho_be

In [ ]:
# # ===========================================
# # 🔷 5개 Stack 모델을 Soft Voting으로 결합
# # ===========================================

# from sklearn.ensemble import VotingClassifier

# # VotingClassifier는 원래 predict_proba를 지원하는 모델만 가능
# # 우리 스택 모델들은 모두 predict_proba 지원하므로 문제 없음
# weights_desc = '''
# | 스택                         | 추천 이유     | weight |
# | -------------------------- | --------- | ------ |
# | Stack #1 (Boosting+LR)     | 대부분 최고 성능 | **3**  |
# | Stack #3 (CatBoost+GB+SVM) | Recall 강함 | **2**  |
# | Stack #2 (LGBM+RF+MLP)     | 안정적       | **2**  |
# | Stack #4 (XGB+LR+SVM)      | 선형 경계 보정  | **1**  |
# | Stack #5 (RF+SGD+MLP)      | 편향 다양성 확보 | **1**  |
# '''

# # Stack 모델 이름 리스트
# # ======================================
# # 🔷 Stacking 모델 pkl 파일명 기반 Voting 리스트
# # ======================================

# stack_model_names = [
#     'stack1(cat+xgb+lgbm)_ho_best',              # ✓ 존재
#     'stack2(lgbm+rf+mlp)_ho_best_smote',         # ✓ 존재
#     'cat+gb+svm_rbf_ho_best',                    # ✓ 존재 (stack3)
#     'stack4(xgb+lsvc+lr)_ho_best',               # ✓ 존재
#     'stack5(rf+sgd+mlp+lr)_ho_best'              # ✓ 존재
# ]


# # 모델 불러오기
# print("📥 저장된 Stack 모델들을 불러오는 중...")
# stacks = []
# for i, model_name in enumerate(stack_model_names, 1):
#     try:
#         model = mu.load_model(model_name)
#         stacks.append((f'stack{i}', model))
#         print(f"✅ stack{i} 로드 완료: {model_name}")
#     except Exception as e:
#         print(f"❌ stack{i} 로드 실패: {e}")
#         raise

# # VotingClassifier 구성
# print("\n🔧 VotingClassifier 구성 중...")
# voting_ensemble = VotingClassifier(
#     estimators=stacks,
#     voting='soft',          # 확률 기반 soft voting
#     weights=[3, 2, 2, 1, 1], # 가중치 
#     n_jobs=-1
# )

# print(f"✅ VotingClassifier 구성 완료 ({len(stacks)}개 스택 모델)")

# # Voting Ensemble 학습 및 평가
# print("\n" + "="*60)
# print("🚀 Voting Ensemble 학습 시작")
# print("="*60)

# ret_result = uu.get_model_train_eval(
#     voting_ensemble, 
#     'voting_ensemble_5stacks', 
#     X_train, X_test, y_train, y_test
# )

# print("\n📊 Voting Ensemble 결과:")
# print(ret_result['result_dict'])

In [ ]:
# 시각화
mo.model_metrics_graph(results, 'LGBM 데이터별 성능지표 비교')